# LLaMA Factory 推理与评估实验

> 本 Notebook 演示基于 LLaMA Factory 框架的三种核心功能：  
> 1. **对话推理**：使用微调后的 LoRA 模型对服装属性标签进行文本描述生成  
> 2. **训练前预测**：在模型微调前执行 predict 任务，获取基线性能指标  
> 3. **训练后预测**：在模型微调后执行 predict 任务，评估微调带来的性能提升

## 一、对话推理测试

使用 `llamafactory-cli chat` 命令加载已微调的 LoRA 模型，对多条服装属性标签（格式：`属性类型#属性值*...`）进行批量推理，模型将根据属性标签生成对应的自然语言商品描述文本。

**配置文件**：`lora_LlamaFactory_infer.yaml`（指定基础模型路径、LoRA adapter 权重路径及推理超参数）

In [1]:
#测试输入
# "类型#裙*版型#宽松*版型#显瘦*图案#刺绣*裙长#连衣裙*裙衣长#中长款*裙款式#拼接*裙款式#亮片" 
# "类型#裤*版型#显瘦*风格#简约*裤款式#破洞*裤款式#流苏" 
# "类型#上衣*图案#格子*图案#刺绣*图案#撞色*衣样式#衬衫" 
!echo -e "类型#裙*版型#宽松*版型#显瘦*图案#刺绣*裙长#连衣裙*裙衣长#中长款*裙款式#拼接*裙款式#亮片\nexit" | llamafactory-cli chat lora_LlamaFactory_infer.yaml

[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:15:59,958 >> loading file tokenizer.model
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:15:59,958 >> loading file tokenizer.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:15:59,958 >> loading file added_tokens.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:15:59,958 >> loading file special_tokens_map.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:15:59,958 >> loading file tokenizer_config.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:15:59,958 >> loading file chat_template.jinja
[INFO|tokenization_utils_base.py:2337] 2026-07-10 22:16:00,290 >> Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
[INFO|configuration_utils.py:763] 2026-07-10 22:16:00,296 >> loading configuration file ./DeepSeekR1DistillLlama8B/config.json
[INFO|configuration_utils.py:839] 2026-07-10 22:16:00,297 >> Model config LlamaConfig {
  "architect

## 二、训练前预测评估

在执行 LoRA 微调**之前**，使用 `llamafactory-cli train` 以 `predict` 模式对测试集进行推理，  
将预测结果保存到指定输出目录，作为 **基线（Baseline）** 性能指标，便于与训练后结果进行对比。

**配置文件**：`lora_LlamaFactory_predict_BeforeTrain.yaml`（`do_predict: true`，`do_train: false`）

> ⚠️ **注意：输出结果包含 `<think>` 推理链标签**  
> DeepSeekR1DistillLlama8B 在 `model.generate()` 阶段会先输出 `<think>...</think>` 推理过程，再生成最终答案。  
> LlamaFactory 在计算 ROUGE 时直接对**完整生成序列**（含推理链）与参考答案对比，导致 ROUGE-L 被严重低估。  
> 如需获取剔除推理链后的真实 ROUGE 分数，请参考 `6.4.ROUGE_Eval_StripThink_DeepSeekR1DistillLlama8B.ipynb`。

In [2]:
# ─────────────────────────────────────────────────────────────────
# 功能：执行训练前预测（Baseline Evaluation）
# 命令：llamafactory-cli train
#   - 子命令 train 用于启动训练/评估流程；当配置文件中设置
#     do_train: false / do_predict: true 时，跳过训练步骤，
#     仅对测试集执行前向推理并将预测结果写入输出目录。
# 参数：lora_LlamaFactory_predict_BeforeTrain.yaml
#   - 指定基础模型路径、LoRA adapter（若有）、测试集路径、
#     输出目录、评估指标（BLEU/ROUGE）等配置项。
# 预期输出：output_dir 下生成 generated_predictions.txt（每行为模型预测文本）
#           及 predict_results.json（含 BLEU-4、ROUGE-L 等量化指标）
# ─────────────────────────────────────────────────────────────────
!llamafactory-cli train lora_LlamaFactory_predict_BeforeTrain.yaml  # ! 前缀：在 Jupyter 中以 shell 子进程执行该命令；返回值为命令退出码（0 表示成功）

[INFO|2026-07-10 22:16:13] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:16:13,703 >> loading file tokenizer.model
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:16:13,703 >> loading file tokenizer.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:16:13,703 >> loading file added_tokens.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:16:13,703 >> loading file special_tokens_map.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:16:13,703 >> loading file tokenizer_config.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:16:13,703 >> loading file chat_template.jinja
[INFO|tokenization_utils_base.py:2337] 2026-07-10 22:16:14,065 >> Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
[INFO|configuration_utils.py:763] 2026-07-10 22:16:14,074 

## 三、训练后预测评估

在完成 LoRA 微调**之后**，使用 `llamafactory-cli train` 以 `predict` 模式对相同测试集进行推理，  
将预测结果与「二、训练前预测评估」的结果进行对比，量化微调带来的性能提升（BLEU-4、ROUGE-L 等指标）。

**配置文件**：`lora_LlamaFactory_predict_AfterTrain.yaml`（加载已保存的微调后 LoRA checkpoint）

> ✅ **注意：训练后预测结果已不含 `<think>` 推理链标签**  
> 经过 LoRA 微调后，模型输出已收敛至目标任务格式，`model.generate()` 不再产生 `<think>...</think>` 推理链，直接生成目标文本。  
> 因此本节的 ROUGE-L 可直接反映微调后模型的真实生成质量，无需额外剔除推理链即可与参考答案进行对比。  
> 与「二、训练前预测评估」（含推理链，ROUGE-L 被低估）的对比更能体现微调带来的实际性能提升。

In [3]:
# ─────────────────────────────────────────────────────────────────
# 功能：执行训练后预测（Post-Training Evaluation）
# 命令：llamafactory-cli train
#   - 与训练前预测相同，以 predict 模式运行；此时配置文件中
#     adapter_name_or_path 指向微调完成后保存的 LoRA checkpoint，
#     模型已经过数据集微调，预期生成质量显著优于训练前基线。
# 参数：lora_LlamaFactory_predict_AfterTrain.yaml
#   - 加载微调后的 LoRA adapter 权重，其余配置（测试集、输出目录、
#     评估指标）与训练前配置保持一致，确保对比实验的公平性。
# 预期输出：output_dir 下生成 generated_predictions.txt 及
#           predict_results.json，将其与训练前结果对比以评估微调效果
# ─────────────────────────────────────────────────────────────────
!llamafactory-cli train lora_LlamaFactory_predict_AfterTrain.yaml  # ! 前缀：在 Jupyter 中以 shell 子进程执行该命令；返回值为命令退出码（0 表示成功）

[INFO|2026-07-10 22:54:09] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:54:09,793 >> loading file tokenizer.model
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:54:09,793 >> loading file tokenizer.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:54:09,793 >> loading file added_tokens.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:54:09,793 >> loading file special_tokens_map.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:54:09,793 >> loading file tokenizer_config.json
[INFO|tokenization_utils_base.py:2066] 2026-07-10 22:54:09,793 >> loading file chat_template.jinja
[INFO|tokenization_utils_base.py:2337] 2026-07-10 22:54:10,134 >> Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
[INFO|configuration_utils.py:763] 2026-07-10 22:54:10,141 